In [7]:
from pathlib import Path
import sys
from collections import Counter
import json
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd()

    start = start.resolve()

    for path in [start, *start.parents]:
        if (path / "data" / "processed" / "test.jsonl").exists() and (path / "src").exists():
            return path

    raise FileNotFoundError("Could not find project root with data/processed/test.jsonl and src/")


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

def read_jsonl(path: Path) -> list[dict]:
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))

    return rows

def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

PRED_PATH = PROJECT_ROOT / "qlora_manual_outputs" / "predictions" / "qlora_manual_token_aware_v1_test.jsonl"
GOLD_PATH = PROJECT_ROOT / "data" / "processed" / "test.jsonl"
METRICS_PATH = PROJECT_ROOT / "reports" / "qlora_manual_metrics.json"

predictions = read_jsonl(PRED_PATH)
gold_rows = read_jsonl(GOLD_PATH)
metrics = read_json(METRICS_PATH)

assert len(predictions) == len(gold_rows)

print(f"Predictions: {len(predictions)}")
print(f"Gold rows: {len(gold_rows)}")
print(f"Metrics: {metrics}")

Project root: C:\Users\Артём\PycharmProjects\routing-support-tickets
Predictions: 150
Gold rows: 150
Metrics: {'ticket_type_accuracy': 0.78, 'ticket_type_macro_f1': 0.6615530303030304, 'topic_accuracy': 0.42, 'topic_macro_f1': 0.3260079051383399, 'urgency_accuracy': 0.43333333333333335, 'urgency_macro_f1': 0.32370930955836613, 'tag_precision': 0.6666666666666666, 'tag_recall': 0.5748344370860927, 'tag_f1': 0.6173541963015649, 'exact_match': 0.013333333333333334, 'exact_match_without_tags': 0.17333333333333334, 'json_validity': 1.0, 'schema_validity': 1.0}


In [9]:
topic_counter = Counter()

for pred in predictions:
    parsed = pred["parsed_response"]

    if not isinstance(parsed, dict):
        continue

    topic = parsed.get("topic", "__MISSING__")
    topic_counter[topic] += 1

print("Most common predicted topics: \n")

for topic, count in topic_counter.most_common():
    print(f"{count:>3} | {topic}")

Most common predicted topics: 

 96 | Technical Support
 19 | Product Support
 11 | Billing and Payments
 10 | Customer Service
  9 | Service Outages and Maintenance
  3 | Sales and Pre-Sales
  1 | Returns and Exchanges
  1 | IT Support
